# How to Run Ingest a discourse forum into the Weaviate database in the Cloud


In [ ]:
# %pip install --upgrade pip
# %pip install weaviate-client
# %pip install langchain==0.3.20
# %pip install openai==1.65.4
# %pip install langchain-openai==0.3.7
# %pip install langchain-weaviate==0.0.4
# %pip install langchain-community==0.3.19
# %pip install pymupdf
# %pip install python-dotenv
# %pip install tdqm

In [ ]:
# Use LSST kernel and install weaviate
# %pip install --upgrade pip
# %pip install weaviate-client

In [ ]:
import json
import os
from pathlib import Path
import time
import requests
from tqdm import tqdm
from urllib.parse import urljoin

import weaviate
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents.base import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_weaviate.vectorstores import WeaviateVectorStore
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

In [ ]:
load_dotenv()

## Scrape data from the forum

In [ ]:
# Read in API key for forum
forum_username = 'leanne'
forum_key = os.getenv("COMMUNITY_API_KEY")

In [ ]:
# Configuration
DISCOURSE_URL = "https://community.lsst.org/" 
HEADERS = {
    "Accept": "application/json",
    # Uncomment and set your API key/username if the forum requires it:
    "Api-Key": forum_key,
    "Api-Username": forum_username
}

In [ ]:
# Test connection
page = 0
url = f"{DISCOURSE_URL}/latest.json?page={page}"
print(f"Fetching {url}")
response = requests.get(url, headers=HEADERS)
#response.json()
assert response.raise_for_status() is None

### Scrape the whole site and write to JSON

In [ ]:
def count_all_pages(discourse_url):
    """
    Counts the total number of pages on the fourm.

    Args:
        discourse_url (type): URL of the discourse forum

    Returns:
        int: The number of pages
    """
    pass
    HEADERS = {
        "Accept": "application/json",
    }    
    page_count = 1
    next_url = f"{DISCOURSE_URL}/latest.json"

    while True:
        response = requests.get(next_url, headers=HEADERS)
        if response.status_code != 200:
            # raise Exception(f"Error fetching page {page_count}: {response.status_code}")
            print(f"Error fetching page {page_count}: {response.status_code}")
            continue
        
        data = response.json()
        more_url = data.get("topic_list", {}).get("more_topics_url")
        
        # print(f"✅ Fetched page {page_count}")
        if not more_url:
            break
        
        next_url = urljoin(DISCOURSE_URL, more_url)
        page_count += 1

    print(f"\nTotal pages found: {page_count}")
    return page_count

In [ ]:
def get_latest_topics(page):
    """
    Get the latest topics on a page

    Args:
        page (str): The page

    Returns:
        int: The list of topics
    """
    url = f"{DISCOURSE_URL}/latest.json?page={page}"
    response = requests.get(url, headers=HEADERS)
    if response.status_code != 200:
        print(f"Failed to fetch page {page}: {response.status_code}")
        return []
    data = response.json()
    return data.get("topic_list", {}).get("topics", [])

In [ ]:
def get_posts_for_topic(topic_id):
    """
    Get all posts for a topic topics

    Args:
        discourse_url (type): URL of the discourse forum
        topic_id (int): The topic ID

    Returns:
        dict: The posts
    """
    url = f"{DISCOURSE_URL}/t/{topic_id}.json"
    response = requests.get(url, headers=HEADERS)
    if response.status_code != 200:
        raise Exception(f"HTTP {response.status_code}")
    data = response.json()
    topic_title = data.get("title", f"topic_{topic_id}")
    posts = data.get("post_stream", {}).get("posts", [])
    cleaned_posts = []

    for post in posts:
        cleaned = clean_post(post)
        cleaned_posts.append(cleaned)

    return {
        "topic_id": topic_id,
        "topic_title": topic_title,
        "posts": cleaned_posts
    }

In [ ]:
def clean_post(post):
    """
    Format the data in a post

    Args:
        post (str): The post

    Returns:
        str: The well-formatted post
    """
    return {
        "post_id": post.get("id"),
        "username": post.get("username", "unknown"),
        "created_at": post.get("created_at"),
        "cooked": post.get("cooked", ""),
        "raw": post.get("raw", ""),
    }

In [ ]:
def clean_filename(name):
    """
    Format a filename

    Args:
        name (str): The filename

    Returns:
        str: The well-formatted filename
    """
    return "".join(c if c.isalnum() or c in "._-" else "_" for c in name)

In [ ]:
def scrape_all_topics():
    seen_topic_ids = set()
    for page in tqdm(range(0, MAX_PAGES), desc="Fetching topics"):
        topics = get_latest_topics(page)
        if not topics:
            print("No more topics found. Done.")
            break

        for topic in topics:
            topic_id = topic["id"]
            if topic_id in seen_topic_ids:
                continue
            seen_topic_ids.add(topic_id)

            try:
                topic_data = get_posts_for_topic(topic_id)
                filename = f"{topic_id}_{clean_filename(topic_data['topic_title'][:50])}.json"
                path = os.path.join(OUTPUT_DIR, filename)
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(topic_data, f, indent=2, ensure_ascii=False)
                time.sleep(SLEEP_TIME)
            except Exception as e:
                print(f"Error fetching topic {topic_id}: {e}")
                continue

In [ ]:
def scrape_and_aggregate():
    all_posts = []
    seen_topic_ids = set()

    for page in tqdm(range(0, MAX_PAGES), desc="Fetching topics"):
        topics = get_latest_topics(page)
        if not topics:
            print("No more topics found.")
            break

        for topic in topics:
            topic_id = topic["id"]
            if topic_id in seen_topic_ids:
                continue
            seen_topic_ids.add(topic_id)

            try:
                posts = get_posts_for_topic(topic_id)
                all_posts.extend([posts])
                time.sleep(SLEEP_TIME)
            except Exception as e:
                print(f"Error fetching topic {topic_id}: {e}")
                continue
                
    # Write everything to a single file
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_posts, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved {len(posts)} posts to {OUTPUT_FILE}")

    return all_posts

In [ ]:
# Run scrape and dump to JSON
OUTPUT_DIR = "discourse_export"
OUTPUT_FILE = OUTPUT_DIR + "/all_discourse_posts.json"
SLEEP_TIME = 1.0  # Delay to avoid rate-limiting. 0.5 gives HTTP 429 too many requests error. 1.5 is long.  
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Run this to get total num pages  and then set 
#MAX_PAGES = count_all_pages(DISCOURSE_URL)
MAX_PAGES = 200  # Set high for whole site (count_all_pages == 110)
print(f"Total number of pages to scrape {MAX_PAGES}")

In [ ]:
# Test with small number -- writes all to individual files 
MAX_PAGES = 2 
scrape_all_topics()

In [ ]:
# scrape and aggregate into one json and write to 1 file and return object
# Step 1: Scrape
t1 = time.time()
posts = scrape_and_aggregate()
t2 = time.time()
print(f"Scraped {MAX_PAGES} pages in {t2 - t1:.2f}s")

## Load into weaviate database

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_api_key

In [ ]:
# Required configuration
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

In [ ]:
# Open to weaviate client 
client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,  # Default is 80, WCD uses 443
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,  # Default is 50051, WCD uses 443
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),  # The API key to use for authentication
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
)

print("Client is live:", client.is_live())
print(client.collections.list_all().keys())  # List all collections in database
# print(client.collections.get("collection_name")) # View the configuration of a collection
# client.collections.delete("collection_name")  # THIS WILL DELETE THE SPECIFIED COLLECTION AND ITS OBJECTS
client.close()